## Setup
* data pipeline a5
* Text Embedding에서 query랑 document 컬럼을 나누어 임베딩 (query=사고원인, document=사고원인,타겟)
* numerical embedding 제거, categorical embedding 제거 (인적사고, 물적사고는 둘다 제외 임베딩 모델이 학습함)
* tfidf ngram_range 1로 지정 (ngram 2 는 의미가 모호함 모든 group str 다 concat치기 때문에)
* weight text: 0.85 context: 0.15
* corpus 데이터프레임 분리
* reranking 과정에서 각 retrieval 결과 합연산
* p4와 같으나 기록이 아닌 사례로 바꿈
* retrieval label 생성을 모델을 통해 다시 하도록 변경
* generation 파라미터 변경 (성능 변화 없음)
* text generator 파라미터 리팩토링
* llm dtype -> bfloat16으로 변경
* generation evaluator에서 text 전처리 부분 제거 (output 형식자체도 평가하기 위함)
* prompt에 부위 컬럼 추가 (중요 키워드가 제공되는 경우가 있음)
* 6-4 에서 코드 리팩토링
* embedding context 구성 c1 사용 (embedding 모델 구성 데이터도 포맷팅함)
* 프롬프트 마지막 생성 시 '이번' 추가
* label 생성 로직 변경 (사고원인 + target encoding with BGE-m3)
* 프롬프트 변경
* context size 및 generation size 변경 (3072 -> 2048, 512 -> 256)
* 사고원인, target 200자 제한 + (프롬프트 200자 이내로 변경)
* GPT 프롬프트 엔지니어링
* prompt 내 '대책'을 '재발방지대책 및 향후조치계획' 으로 수정
* 외부문서 활용 + (프롬프트 내 환경 정보 '인적사고', '계절/기온' 제거)
* 프롬프트 변경 + static thresholding + '사례N.' 에서 '사례N)' 으로 변경
* dpr 에도 max len 추가
* context 나열 순서 변경 (시간 -> 공간 -> 원인 -> 재발방지대책)
* prompt 명확화
* prompt 입력 프로세스 변경

In [1]:
SEED = 42
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import pandas as pd
import faiss
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import gc
from utils import *

import torch
from sentence_transformers import SentenceTransformer
from vllm import LLM, SamplingParams

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'  # Ubuntu
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/hdd_data2/yjk/miniconda3/envs/vllm-073/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 03-15 20:57:28 [__init__.py:256] Automatically detected platform cuda.


2025-03-15 20:57:28,325	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


device(type='cuda')

In [2]:
class CFG:
    debug = False
    dp_version = "a5"
    ext_version = "e1"
    model = "rag"
    model_version = "llama3.1-8b_bS11-1_p17"
    drop_vars = ["발생일시", "사고인지_일시", "사고인지_월", "사고인지_시간"]
    target_var = "재발방지대책 및 향후조치계획"
    if debug:
        llm_model_id = "/hdd_data2/llm_archive/gemma-2-2b-it"
        embedding_model_id = "/hdd_data2/llm_archive/bge-m3"
    else:
        llm_model_id = "/hdd_data2/llm_archive/Llama-3.1-8B-Instruct"
        embedding_model_id = "/hdd_data2/llm_archive/bge-m3"
    batch_size = 256
    tfidf_ngram_range = (1, 1)
    embedding_max_length = 1024
    retrieval_weights = {"text": 0.85, "context": 0.15}
    static_threshold = 0.5
    retrieval_top_k = 50
    reranking_top_k = 5
    llm_params = {
        "model": llm_model_id,
        "max_model_len": 2048, "quantization": "fp8", "load_format": "auto", "dtype": "bfloat16",
        "gpu_memory_utilization": 0.2 if debug else 0.8, "tensor_parallel_size": 1, "distributed_executor_backend": "ray",
        "enforce_eager": True, "seed": SEED
    }
    sampling_params = {"max_tokens": 256, "temperature": 0.5, "top_p": 0.95, "top_k": 40}
    replace_str = "없음"
    context_max_len = 300

In [3]:
architecture_path = f"architecture/{CFG.model}_{CFG.model_version}"
createFolder(architecture_path)
seed_everything(SEED)

prompt_template = {
    "user": """
## 작업설명
기본규정 및 과거사례를 참고하여 이번 사례에 적합한 재발방지대책 및 향후조치계획을 200자 이내로 작성하세요.
기본규정을 우선적으로 참고하여, 이번 사례의 사고원인과 일치하는 사고원인이 있을 경우 해당 규정의 재발방지대책 및 향후조치계획을 그대로 작성하세요.  
기본규정에서 일치하는 사고원인이 없을 경우, 과거사례에서 유사한 사고 사례를 찾아 해당 사례의 재발방지대책 및 향후조치계획을 종합하여 작성하세요.  
제공된 정보 외에 새로운 정보를 추가하지 마세요.  
콤마로 구분된 일반 텍스트 형식으로 작성하며, Markdown 형식을 사용하지 마세요.

## 기본규정
사고원인: 기타 or 모름 or 알 수 없음 or 작업자의 과실 및 실수 or 작업자의 부주의 및 관리소홀
재방지대책 및 향후조치계획: 작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.

## 과거사례
{contexts}

## 이번사례
계절/기온: {season}
위치: {location}
사고유형: {type}
사고원인: {query}
재방지대책 및 향후조치계획:
"""
}

## Loading data

In [4]:
df_full = pickleIO(None, f"dataset/prepdata/{CFG.dp_version}/df_full.pkl", "r")
df_valid = pickleIO(None, f"dataset/prepdata/{CFG.dp_version}/df_valid.pkl", "r")
df_test = pickleIO(None, f"dataset/prepdata/{CFG.dp_version}/df_test.pkl", "r")
# df_ext = pd.concat([pd.read_csv(f'dataset/extdata/{CFG.ext_version}/external_docs_qa.csv'), pd.read_csv(f'dataset/extdata/{CFG.ext_version}/external_docs_qa_qwen.csv')], axis=0)
# df_ext["사고원인"] = df_ext["사고원인"].str.strip()
# df_ext[CFG.target_var] = df_ext[CFG.target_var].str.strip()
# df_ext = df_ext[df_ext["사고원인"].apply(lambda x: "사고원인" not in x)].drop_duplicates(subset=["사고원인"]).reset_index(drop=True)

In [5]:
df_full = df_full.drop(CFG.drop_vars, axis=1)
df_valid = df_valid.drop(CFG.drop_vars, axis=1)
df_test = df_test.drop(CFG.drop_vars, axis=1)

In [6]:
# add '부위'
df_full["부위"] = df_full["부위_level0"] + " " + df_full["부위_level1"]
df_valid["부위"] = df_valid["부위_level0"] + " " + df_valid["부위_level1"]
df_test["부위"] = df_test["부위_level0"] + " " + df_test["부위_level1"]

In [7]:
df_full.shape, df_valid.shape, df_test.shape

((21604, 36), (1000, 34), (964, 31))

In [8]:
if CFG.debug:
    df_full = df_full.iloc[:100]
    df_valid = df_valid.iloc[:10]
    df_test = df_test.iloc[:10]
# df_corpus = pd.concat([df_full, df_ext[["사고원인", CFG.target_var]]], axis=0).fillna(CFG.replace_str).reset_index(drop=True)
df_corpus = pd.concat([df_full], axis=0).fillna(CFG.replace_str).reset_index(drop=True)

In [9]:
df_full.shape, df_valid.shape, df_test.shape

((21604, 36), (1000, 34), (964, 31))

In [10]:
df_full["부위"]

0           기타 앞
1          자재 바닥
2          공구류 앞
3          자재 지하
4         비계 상부위
          ...   
21599      자재 바닥
21600    방음벽 잔재물
21601      자재 바닥
21602     기타 상부위
21603      기타 내부
Name: 부위, Length: 21604, dtype: object

In [11]:
# df_full

In [12]:
print(df_full.columns)

embed_info = {
    "num": [],
    "cat": [],
    "group": ["공사종류", "공종", "사고객체", "작업프로세스", "장소"],
    "query_str": ["계절/기온", "부위", "인적사고", "사고원인"],
    "doc_str": ["사고원인", CFG.target_var],
    "contexts_vars": ["계절/기온", "부위", "인적사고", "사고원인", CFG.target_var],
}

embed_container = {
    "full": {},
    "valid": {},
    "test": {},
}

Index(['날씨', '기온', '습도', '연면적', '인적사고', '물적사고', '사고원인', '재발방지대책 및 향후조치계획',
       'aug_사고원인', 'aug_재발방지대책 및 향후조치계획', '발생일시_월', '발생일시_시간', '사고인지_시기', '계절',
       '계절/기온', '공사종류_level0', '공사종류_level1', '공사종류_level2', '공사종류_level3',
       '지상층', '지하층', '공종_level0', '공종_level1', '사고객체_level0', '사고객체_level1',
       '작업프로세스_level0', '작업프로세스_level1', '장소_level0', '장소_level1', '장소_level2',
       '장소_level3', '부위_level0', '부위_level1', '부위_level2', '부위_level3', '부위'],
      dtype='object')


In [13]:
setlist = []
for col in embed_info["group"]:
    if col not in ["인적사고", "물적사고"]:
        sub_setlist = []
        for sub_col in df_full.filter(regex=f"^{col}").columns:
            sub_setlist.extend(list(df_full[sub_col].unique()))
        setlist.append(set(sub_setlist))
    else:
        setlist.append(set(df_full[col].unique()))
u = set.intersection(*setlist)
u

{'기타'}

## Create Vector Store & Evaluator

### Define Embedding Models

In [14]:
class TextEmbedding():
    def __init__(self, query_str_vars, doc_str_vars, model_id, max_length, batch_size, device, max_len=200):
        self.query_str_vars = query_str_vars
        self.doc_str_vars = doc_str_vars
        self.separator = "\n"
        self.model_id = model_id
        self.model = SentenceTransformer(model_id, tokenizer_kwargs={"max_length": max_length}, device=device)
        self.batch_size = batch_size
        self.db = None
        self.max_len = max_len

    def create_contexts(self, row: pd.Series):
        tmp = []
        for col in row.keys():
            if col == "인적사고":
                tmp.append(f"{'사고유형'}: {row[col][:self.max_len]}")
            elif col == "부위":
                tmp.append(f"{'위치'}: {row[col][:self.max_len]}")
            else:
                tmp.append(f"{col}: {row[col][:self.max_len]}")
        return "\n".join(tmp)
    
    def fn_embed(self, x):
        return self.model.encode(x, batch_size=self.batch_size, normalize_embeddings=True)
    
    def embedding(self, data, mode):
        embed = {}
        embed[mode] = self.fn_embed(data[(self.query_str_vars if mode == "query" else self.doc_str_vars)].apply(self.create_contexts, axis=1))
        return embed

    def create_db(self, data):
        embed = self.embedding(data, mode="doc")
        embed = np.concatenate(list(embed.values()), axis=1)
        self.db = faiss.IndexFlatIP(embed.shape[1])
        self.db.add(embed)

    def search(self, query, k=10):
        query = self.embedding(query, mode="query")
        query = np.concatenate(list(query.values()), axis=1)
        scores, indices = self.db.search(query, k=k)
        return {"scores": scores, "indices": indices}

class ContextEmbedding():
    def __init__(self, num_vars, cat_vars, group_vars, ngram_range=(1, 2)):
        self.num_vars = num_vars
        self.cat_vars = cat_vars
        self.group_vars = group_vars
        # TabularEmbedding 관련 변수들
        self.scaler = MinMaxScaler()
        self.encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype="float32")
        # GroupEmbedding 관련 변수들
        self.tfidf = TfidfVectorizer(min_df=3, ngram_range=ngram_range, dtype="float32")
        self.nan_str = ["", "-", "없음"]
        self.replace_str = "없음"
        self.db = None
    
    def group_preprocess(self, x):
        return " ".join([i for i in x if (i not in self.nan_str) and (i != self.replace_str)])
    
    def fit(self, data):
        # TabularEmbedding fit
        if len(self.num_vars) > 0:
            self.scaler.fit(data[self.num_vars])
        if len(self.cat_vars) > 0:
            self.encoder.fit(data[self.cat_vars])
        # GroupEmbedding fit
        if len(self.group_vars) > 0:
            sentences = []
            for col in self.group_vars:
                if col == "인적사고":
                    sentences.append(data[col].apply(lambda x: "".join(x.split())))
                else:
                    sentences.append(data.filter(regex=f"^{col}_").apply(self.group_preprocess, axis=1))
            sentences = pd.DataFrame(sentences).apply(lambda x: " ".join(x), axis=0).to_list()
            self.tfidf.fit(sentences)
    
    def embedding(self, data):
        embed = {}
        # TabularEmbedding 임베딩
        if len(self.num_vars) > 0:
            embed["num"] = self.scaler.transform(data[self.num_vars])
        if len(self.cat_vars) > 0:
            embed["cat"] = self.encoder.transform(data[self.cat_vars])
        # GroupEmbedding 임베딩
        if len(self.group_vars) > 0:
            sentences = []
            for col in self.group_vars:
                if col == "인적사고":
                    sentences.append(data[col].apply(lambda x: "".join(x.split())))
                else:
                    sentences.append(data.filter(regex=f"^{col}_").apply(self.group_preprocess, axis=1))
            sentences = pd.DataFrame(sentences).apply(lambda x: " ".join(x), axis=0).to_list()
            embed["group"] = self.tfidf.transform(sentences).toarray().astype("float32")
        return embed
    
    def create_db(self, data):
        embed = self.embedding(data)
        embed = np.concatenate(list(embed.values()), axis=1)
        self.db = faiss.IndexFlatIP(embed.shape[1])
        self.db.add(embed)
    
    def search(self, query, k=10):
        query = self.embedding(query)
        query = np.concatenate(list(query.values()), axis=1)
        scores, indices = self.db.search(query, k=k)
        return {"scores": scores, "indices": indices}

### Define Retrieval & Generation Evaluator

In [15]:
class RetrievalEvaluator():
    def __init__(self, k_range=[1, 3, 5]):
        self.k_range = k_range

    def __call__(self, preds, gts):
        eval_results = {}
        for retrieval_type in ["text", "context", "reranking"]:
            eval_results[retrieval_type] = {}
            for k in self.k_range:
                eval_results[retrieval_type][f"recall@{k}"] = []
                eval_results[retrieval_type][f"precision@{k}"] = []
                eval_results[retrieval_type][f"accuracy@{k}"] = []
                for y_pred, y_true in zip(preds[retrieval_type]["indices"], gts):
                    # accuracy@k 계산
                    accuracy = float(y_true.split(",")[0] in set(map(str, y_pred[:k])))
                    eval_results[retrieval_type][f"accuracy@{k}"].append(accuracy)
                    # recall & precision
                    y_true = set(map(str, y_true.split(",")))
                    y_pred = set(map(str, y_pred[:k]))
                    # recall@k 계산
                    recall = (len(y_true & y_pred) / len(y_true)) if len(y_true) > 0 else 0
                    eval_results[retrieval_type][f"recall@{k}"].append(recall)
                    # precision@k 계산 
                    precision = (len(y_true & y_pred) / k) if k > 0 else 0
                    eval_results[retrieval_type][f"precision@{k}"].append(precision)
            eval_results[retrieval_type] = pd.DataFrame(eval_results[retrieval_type])
        return eval_results

class GenerationEvaluator():
    def __init__(self, batch_size, device):
        self.model = SentenceTransformer('jhgan/ko-sbert-sts', tokenizer_kwargs={"max_length": 512}, device=device, use_auth_token=False)
        self.batch_size = batch_size

    @staticmethod
    def cosine_similarity(a, b):
        dot_product = np.dot(a, b)
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)
        return dot_product / (norm_a * norm_b) if norm_a != 0 and norm_b != 0 else 0
    
    @staticmethod
    def jaccard_similarity(text1, text2):
        """자카드 유사도 계산"""
        set1, set2 = set(text1.split()), set(text2.split())  # 단어 집합 생성
        intersection = len(set1.intersection(set2))  # 교집합 크기
        union = len(set1.union(set2))  # 합집합 크기
        return intersection / union if union != 0 else 0

    def __call__(self, preds, gts):
        sample_scores = []
        preds_embed = self.model.encode(preds, batch_size=self.batch_size)
        gts_embed = self.model.encode(gts, batch_size=self.batch_size)
        for pred, gt, pred_embed, gt_embed in zip(preds, gts, preds_embed, gts_embed):
            sample_score = self.cosine_similarity(pred_embed, gt_embed) * 0.7 + self.jaccard_similarity(pred, gt) * 0.3
            sample_score = max(sample_score, 0)
            sample_scores.append(sample_score)
        return np.mean(sample_scores)

In [16]:
text_embedding = TextEmbedding(embed_info["query_str"], embed_info["doc_str"], model_id=CFG.embedding_model_id, max_length=CFG.embedding_max_length, batch_size=CFG.batch_size, device=device, max_len=CFG.context_max_len)
text_embedding.create_db(df_corpus)

context_embedding = ContextEmbedding(embed_info["num"], embed_info["cat"], embed_info["group"], ngram_range=CFG.tfidf_ngram_range)
context_embedding.fit(df_full)
context_embedding.create_db(df_corpus)

/hdd_data2/yjk/miniconda3/envs/vllm-073/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


In [17]:
retrieval_evaluator = RetrievalEvaluator()
generation_evaluator = GenerationEvaluator(batch_size=CFG.batch_size, device=device)

/hdd_data2/yjk/miniconda3/envs/vllm-073/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py:195: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


In [18]:
def create_labels(target, corpus):
    labels = []
    model_sts = SentenceTransformer("/hdd_data2/llm_archive/bge-m3/", tokenizer_kwargs={"max_length": 1024})
    target_embed = model_sts.encode(target, batch_size=CFG.batch_size, normalize_embeddings=True)
    corpus_embed = model_sts.encode(corpus, batch_size=CFG.batch_size, normalize_embeddings=True)
    scores = ((target_embed @ corpus_embed.T) + 1) / 2
    indices = np.argsort(scores, axis=1)[:, ::-1]
    labels = pd.DataFrame(indices[:, :CFG.reranking_top_k], dtype="str").apply(lambda x: ",".join(x), axis=1)
    del model_sts
    gc.collect()
    torch.cuda.empty_cache()
    return labels.to_list()

In [19]:
df_valid["labels"] = create_labels(
    df_valid[["사고원인", CFG.target_var]].apply(lambda x: "\n".join(x), axis=1).to_list(),
    df_corpus[["사고원인", CFG.target_var]].apply(lambda x: "\n".join(x), axis=1).to_list(),
)
df_valid["scores"] = 1.0
df_valid["labels"].head()

0      131,1088,11008,19545,12811
1    19616,19038,2589,14695,17606
2       11809,18968,17586,345,217
3       19951,5778,7121,3068,1295
4       16402,6833,4672,1176,4960
Name: labels, dtype: object

## Define RAG Pipeline

### Define Text Generator

In [20]:
class TextGenerator():
    def __init__(self, llm_params, sampling_params):
        self.llm = LLM(**llm_params)
        self.sampling_params = SamplingParams(**sampling_params)

    def create_prompts(self, sources, prompt_template):
        prompts = []
        for prompt in sources:
            prompt["contexts"] = "\n".join([f"사례{i+1}) {context}" for i, context in enumerate(prompt["contexts"])])
            if "system" in prompt_template.keys():
                msg = [
                    {"role": "system", "content": prompt_template["system"].strip()},
                    {"role": "user", "content": prompt_template["user"].format(**prompt).lstrip()},
                ]
            else:
                msg = [
                    {"role": "user", "content": prompt_template["user"].format(**prompt).lstrip()},
                ]
            prompts.append(msg)
        return prompts

    def generate(self, sources, prompt_template):
        prompts = self.create_prompts(sources, prompt_template)
        outputs = self.llm.chat(
            messages=prompts,
            sampling_params=self.sampling_params,
            use_tqdm=True
        )
        gened = []
        for output in outputs:
            gened.append(output.outputs[0].text.strip())
        return {"prompts": [i[-1]["content"] for i in prompts], "generation": gened}

In [21]:
text_generator = TextGenerator(
    llm_params=CFG.llm_params,
    sampling_params=CFG.sampling_params,
)

INFO 03-15 20:59:24 [config.py:583] This model supports multiple tasks: {'score', 'embed', 'reward', 'generate', 'classify'}. Defaulting to 'generate'.
WARNING 03-15 20:59:24 [cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 03-15 20:59:25 [llm_engine.py:235] Initializing a V0 LLM engine (v0.7.4.dev459+g9f374227) with config: model='/hdd_data2/llm_archive/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='/hdd_data2/llm_archive/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=fp8, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_back

2025-03-15 20:59:26,850	INFO worker.py:1841 -- Started a local Ray instance.


INFO 03-15 20:59:27 [ray_distributed_executor.py:176] use_ray_spmd_worker: False
(pid=324493) INFO 03-15 20:59:30 [__init__.py:256] Automatically detected platform cuda.
INFO 03-15 20:59:31 [ray_distributed_executor.py:350] non_carry_over_env_vars from config: set()
INFO 03-15 20:59:31 [ray_distributed_executor.py:352] Copying the following environment variables to workers: ['CUDA_HOME', 'LD_LIBRARY_PATH']
INFO 03-15 20:59:31 [ray_distributed_executor.py:355] If certain env vars should NOT be copied to workers, add them to /hdd_data2/yjk/.config/vllm/ray_non_carry_over_env_vars.json file
INFO 03-15 20:59:31 [cuda.py:285] Using Flash Attention backend.
INFO 03-15 20:59:31 [parallel_state.py:948] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0
INFO 03-15 20:59:31 [model_runner.py:1110] Starting to load model /hdd_data2/llm_archive/Llama-3.1-8B-Instruct...


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:00,  5.57it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:01,  1.86it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  1.54it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.39it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.55it/s]


INFO 03-15 20:59:34 [loader.py:429] Loading weights took 2.73 seconds


INFO 03-15 20:59:35 [model_runner.py:1146] Model loading took 8.4889 GB and 2.870589 seconds
INFO 03-15 20:59:35 [worker.py:267] Memory profiling takes 0.51 seconds
INFO 03-15 20:59:35 [worker.py:267] the current vLLM instance can use total_gpu_memory (44.53GiB) x gpu_memory_utilization (0.80) = 35.62GiB
INFO 03-15 20:59:35 [worker.py:267] model weights take 8.49GiB; non_torch_memory takes 0.21GiB; PyTorch activation peak memory takes 1.18GiB; the rest of the memory reserved for KV Cache is 25.74GiB.
INFO 03-15 20:59:36 [executor_base.py:111] # cuda blocks: 13178, # CPU blocks: 2048
INFO 03-15 20:59:36 [executor_base.py:116] Maximum concurrency for 2048 tokens per request: 102.95x
INFO 03-15 20:59:37 [llm_engine.py:441] init engine (profile, create kv cache, warmup model) took 2.79 seconds


### Define RAG Pipeline

In [22]:
class RAG():
    def __init__(
            self, df_corpus, text_embedding, context_embedding, text_generator, contexts_vars,
            static_threshold=0.5, max_len=200, retrieval_top_k=100, reranking_top_k=5, weights={"text": 1.0, "context": 0.0}
        ):
        self.df_corpus = df_corpus
        self.text_embedding = text_embedding
        self.context_embedding = context_embedding
        self.text_generator = text_generator
        self.contexts_vars = contexts_vars
        self.static_threshold = static_threshold
        self.max_len = max_len
        self.retrieval_top_k = retrieval_top_k
        self.reranking_top_k = reranking_top_k
        self.weights = weights
    
    def reranking(self, searched):
        scores = []
        indices = []
        for text_scores, text_indices, context_scores, context_indices in zip(searched["text"]["scores"], searched["text"]["indices"], searched["context"]["scores"], searched["context"]["indices"]):
            # merge scores
            text_scores = pd.Series(text_scores, index=text_indices)
            text_scores = text_scores[text_scores > self.static_threshold] * self.weights["text"]
            context_scores = pd.Series(context_scores, index=context_indices)
            context_scores = context_scores[context_scores > self.static_threshold] * self.weights["context"]
            reranked_scores = text_scores.add(context_scores, fill_value=0)
            # get top k
            reranked_scores = reranked_scores.sort_values(ascending=False).iloc[:self.reranking_top_k]
            if len(reranked_scores) < self.reranking_top_k:
                reranked_scores = pd.concat([reranked_scores, pd.Series(np.nan, index=[-1] * (self.reranking_top_k - len(reranked_scores)))])
            scores.append(reranked_scores.values)
            indices.append(reranked_scores.index.values)
        scores = np.nan_to_num(np.stack(scores, axis=0), nan=-1)
        indices = np.nan_to_num(np.stack(indices, axis=0), nan=-1)
        return {"scores": scores, "indices": indices}

    def retrieval(self, df_query: pd.DataFrame):
        searched = {
            "text": self.text_embedding.search(df_query, k=self.retrieval_top_k),
            "context": self.context_embedding.search(df_query, k=self.retrieval_top_k),
        }
        # rescaling cosine similarity scores
        searched["text"]["scores"] = (searched["text"]["scores"] + 1) / 2
        searched["context"]["scores"] = (searched["context"]["scores"] + 1) / 2
        # reranking
        searched["reranking"] = self.reranking(searched)
        return searched

    def create_contexts(self, df_selected: pd.DataFrame):
        contexts = []
        for _, row in df_selected.iterrows():
            tmp = []
            for col in self.contexts_vars:
                if col == "인적사고":
                    tmp.append(f"{'사고유형'}: {row[col][:self.max_len]}")
                elif col == "부위":
                    tmp.append(f"{'위치'}: {row[col][:self.max_len]}")
                else:
                    tmp.append(f"{col}: {row[col][:self.max_len]}")
            contexts.append("\n".join(tmp))
        return contexts

    def create_sources(self, df_query: pd.DataFrame, searched: dict):
        sources = []
        for (_, row), searched_indices in zip(df_query.iterrows(), searched["reranking"]["indices"]):
            searched_indices = searched_indices[searched_indices != -1]
            sources.append({
                "contexts": self.create_contexts(self.df_corpus.iloc[searched_indices]) if len(searched_indices) > 0 else [],
                "season": row["계절/기온"],
                "location": row["부위"],
                "type": row["인적사고"],
                "query": row["사고원인"][:self.max_len],
            })
        return sources

    def __call__(self, df_query: pd.DataFrame, prompt_template: dict):
        searched = self.retrieval(df_query)
        sources = self.create_sources(df_query, searched)
        gened = self.text_generator.generate(sources, prompt_template)
        return {"retrieval": searched, "prompts": gened["prompts"], "generation": gened["generation"]}

In [23]:
rag = RAG(df_corpus, text_embedding, context_embedding, text_generator, contexts_vars=embed_info["contexts_vars"], static_threshold=CFG.static_threshold, max_len=CFG.context_max_len, retrieval_top_k=CFG.retrieval_top_k, reranking_top_k=CFG.reranking_top_k, weights=CFG.retrieval_weights)

### Predict & Evaluation

In [24]:
predictions = rag(df_valid, prompt_template)
predictions["generation"][:3]

INFO 03-15 20:59:45 [chat_utils.py:346] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 03-15 21:00:02 [scheduler.py:1769] Sequence group 190 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts: 100%|██████████| 1000/1000 [02:05<00:00,  7.99it/s, est. speed input: 8709.21 toks/s, output: 708.19 toks/s]


['작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.',
 '작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획을 참고합니다.\n\n사고원인이 작업자의 과실 및 실수, 작업자의 부주의 및 관리소홀에 해당하지 않아, 과거사례를 참고하여 유사한 사고 사례를 찾아 종합하여 재발방지대책 및 향후조치계획을 작성합니다.\n\n사례3) 계절/기온: 여름/30°C, 사례4) 계절/기온: 여름/30°C와 유사한 사고 사례가 있습니다. 두 사례 모두 열사병으로 인한 사고였으며, 열사병 방지를 위한 작업장 관리와 작업자 안전교육의 철저함, 충분한 물 제공, 그늘 휴식처 제공, 폭염특보 발효 시 근로자 근무시간 조정 등의 관리 철저가 재발방지대책으로 제시되었습니다.\n\n따라서, 이번 사례의 재발방지대책 및 향후조치계획은 다음과 같습니다.\n\n작업장 관리와 작업자 안전교육의 철저함, 충분한',
 '작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획을 적용합니다.\n\n작업 전 안전교육을 실시하여 작업자들이 안전한 작업 방법을 익히고, 작업관리자 안전점검을 통해 작업장의 위험 요소들을 확인하여 재발 방지 대책을 마련합니다.']

In [25]:
print(predictions["prompts"][0])

## 작업설명
기본규정 및 과거사례를 참고하여 이번 사례에 적합한 재발방지대책 및 향후조치계획을 200자 이내로 작성하세요.
기본규정을 우선적으로 참고하여, 이번 사례의 사고원인과 일치하는 사고원인이 있을 경우 해당 규정의 재발방지대책 및 향후조치계획을 그대로 작성하세요.  
기본규정에서 일치하는 사고원인이 없을 경우, 과거사례에서 유사한 사고 사례를 찾아 해당 사례의 재발방지대책 및 향후조치계획을 종합하여 작성하세요.  
제공된 정보 외에 새로운 정보를 추가하지 마세요.  
콤마로 구분된 일반 텍스트 형식으로 작성하며, Markdown 형식을 사용하지 마세요.

## 기본규정
사고원인: 기타 or 모름 or 알 수 없음 or 작업자의 과실 및 실수 or 작업자의 부주의 및 관리소홀
재방지대책 및 향후조치계획: 작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.

## 과거사례
사례1) 계절/기온: 가을/16°C
위치: 자재 상부위
사고유형: 떨어짐(2미터 미만)
사고원인: 거푸집 해체 작업중 해체된 거푸집을 1.2m 우마에서 거푸집을 내리던중 낙상하여 바닥에 있던 거푸집에 옆구리를 부딪침.
재발방지대책 및 향후조치계획: 사전작업전 안전교육과 자재상태 및 정리정돈을 통한 재발 방지 대책 및 향후 조치 계획.
사례2) 계절/기온: 여름/29°C
위치: 자재 상부위
사고유형: 물체에 맞음
사고원인: 거푸집 해체 작업 도중 부상자가 해체 현장을 지나가다 천장 거푸집에 맞아 부상을 당함
재발방지대책 및 향후조치계획: 거푸집 해체작업 중 신호수 배치 철저와 작업자 건강상태 확인을 포함한 재발 방지 대책 수립 및 이행 감독.
사례3) 계절/기온: 겨울/6°C
위치: 거푸집 상부위
사고유형: 물체에 맞음
사고원인: 3층에서 거푸집 해체공 ***씨가 상부 거푸집을 해체하던중 해체된 거푸집 자재(각재)가 떨어지며앞이 부분에 맞아 치아 손상 됨
재발방지대책 및 향후조치계획: 건설사업관리용역사 및 시공사 안전교육 철저 지시.
사례4) 계

In [26]:
pd.Series([len(i) for i in predictions["generation"]]).describe()

count    1000.000000
mean      151.824000
std       126.203218
min        51.000000
25%        52.000000
50%        68.000000
75%       228.250000
max       504.000000
dtype: float64

In [27]:
np.percentile(pd.Series([len(i) for i in predictions["generation"]]), [75, 95, 99])

array([228.25, 413.15, 461.  ])

In [28]:
print(predictions["generation"][0])

작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.


In [29]:
print(predictions["generation"][1])

작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획을 참고합니다.

사고원인이 작업자의 과실 및 실수, 작업자의 부주의 및 관리소홀에 해당하지 않아, 과거사례를 참고하여 유사한 사고 사례를 찾아 종합하여 재발방지대책 및 향후조치계획을 작성합니다.

사례3) 계절/기온: 여름/30°C, 사례4) 계절/기온: 여름/30°C와 유사한 사고 사례가 있습니다. 두 사례 모두 열사병으로 인한 사고였으며, 열사병 방지를 위한 작업장 관리와 작업자 안전교육의 철저함, 충분한 물 제공, 그늘 휴식처 제공, 폭염특보 발효 시 근로자 근무시간 조정 등의 관리 철저가 재발방지대책으로 제시되었습니다.

따라서, 이번 사례의 재발방지대책 및 향후조치계획은 다음과 같습니다.

작업장 관리와 작업자 안전교육의 철저함, 충분한


In [30]:
print(predictions["generation"][2])

작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획을 적용합니다.

작업 전 안전교육을 실시하여 작업자들이 안전한 작업 방법을 익히고, 작업관리자 안전점검을 통해 작업장의 위험 요소들을 확인하여 재발 방지 대책을 마련합니다.


In [31]:
print(predictions["generation"][3])

작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.


In [32]:
print(predictions["generation"][4])

작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.

작업 전 안전교육 실시를 통해 작업자들이 안전한 작업 방법에 대한 지식을 습득하고, 안전관리자 안전점검을 통해 작업장의 위험 요인을 발견하고 해결할 수 있도록 하여 재발 방지에 기여할 수 있습니다.


In [33]:
df_valid["top_score"] = (text_embedding.fn_embed(df_valid[CFG.target_var].to_list()) * text_embedding.fn_embed(rag.df_corpus[CFG.target_var].iloc[predictions["retrieval"]["reranking"]["indices"][:, 0]].to_list())).sum(axis=1)
df_valid["similarity_difference"] = df_valid["scores"] - df_valid["top_score"]
scores = {
    "generation": generation_evaluator(predictions["generation"], df_valid[CFG.target_var]),
    "retrieval": pd.DataFrame({k: v.mean(axis=0) for k, v in retrieval_evaluator(predictions["retrieval"], df_valid["labels"]).items()}),
    "similarity_difference": df_valid["similarity_difference"].mean(),
}
df_score = {
    "generation": scores["generation"],
    "similarity_difference": scores["similarity_difference"],
}
for k in scores["retrieval"]:
    df_score.update({f"{k}_{metric}": value for metric, value in scores["retrieval"][k].to_dict().items()})
df_score = pd.Series(df_score)  
df_score.loc[["generation", "similarity_difference", "text_accuracy@5", "context_accuracy@5", "reranking_accuracy@5"]]

generation               0.464381
similarity_difference    0.349209
text_accuracy@5          0.254000
context_accuracy@5       0.017000
reranking_accuracy@5     0.240000
dtype: float64

### Save predictions & Evaluation results

In [34]:
y_pred = generation_evaluator.model.encode(predictions["generation"], batch_size=CFG.batch_size, normalize_embeddings=True)
y_true = generation_evaluator.model.encode(df_valid[CFG.target_var].to_list(), batch_size=CFG.batch_size, normalize_embeddings=True)
df_sample = pd.concat([df_valid, pd.DataFrame({"prompts": predictions["prompts"], "generation": predictions["generation"], "similarity": (y_pred * y_true).sum(axis=1)})], axis=1)
df_sample.to_excel(os.path.join(architecture_path, f"{CFG.model}_{CFG.model_version}_predictions.xlsx"), index=False)
with open(os.path.join(architecture_path, f"{CFG.model}_{CFG.model_version}_eval_score.txt"), "w") as f:
    f.write(df_score.to_string())
# df_sample

## Inference

In [35]:
del text_embedding, context_embedding, rag
gc.collect()
torch.cuda.empty_cache()

In [36]:
# df_corpus = pd.concat([df_full, df_valid, df_ext[["사고원인", CFG.target_var]]], axis=0).fillna(CFG.replace_str).reset_index(drop=True)
df_corpus = pd.concat([df_full, df_valid], axis=0).fillna(CFG.replace_str).reset_index(drop=True)

text_embedding = TextEmbedding(embed_info["query_str"], embed_info["doc_str"], model_id=CFG.embedding_model_id, max_length=CFG.embedding_max_length, batch_size=CFG.batch_size, device=device, max_len=CFG.context_max_len)
text_embedding.create_db(df_corpus)

context_embedding = ContextEmbedding(embed_info["num"], embed_info["cat"], embed_info["group"], ngram_range=CFG.tfidf_ngram_range)
context_embedding.fit(pd.concat([df_full, df_valid], axis=0).fillna(CFG.replace_str).reset_index(drop=True))
context_embedding.create_db(df_corpus)

rag = RAG(df_corpus, text_embedding, context_embedding, text_generator, contexts_vars=embed_info["contexts_vars"], retrieval_top_k=CFG.retrieval_top_k, reranking_top_k=CFG.reranking_top_k, weights=CFG.retrieval_weights, max_len=CFG.context_max_len)

/hdd_data2/yjk/miniconda3/envs/vllm-073/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


In [37]:
predictions = rag(df_test, prompt_template)
predictions["generation"][:3]

Processed prompts: 100%|██████████| 964/964 [02:00<00:00,  7.97it/s, est. speed input: 8724.09 toks/s, output: 691.09 toks/s]


['사고원인에 따라 재발방지대책 및 향후조치계획을 다음과 같이 마련합니다.\n\n펌프카 아웃트리거 바닥 고임목을 3단으로 보강한 경우에도 지반 침하가 발생한 점을 고려하여, 사고원인과 유사한 사례 2번의 재발방지대책 및 향후조치계획을 참고하여 다음과 같이 마련합니다.\n\n재발방지대책 및 향후조치계획: 펌프카 설치 위치 사전 검토와 아웃트리거 변위 여부 점검을 통한 재발 방지 대책 마련, 아웃트리거의 펼친 길이가 상이한 점을 확인하여 일관된 펼침 길이를 유지할 수 있도록 조치, 타설 위치가 건물 끝부분 모서리에 위치하는 것을 피하는 조치.',
 '작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획을 적용합니다.',
 '작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획을 적용합니다.']

In [38]:
print(predictions["prompts"][0])

## 작업설명
기본규정 및 과거사례를 참고하여 이번 사례에 적합한 재발방지대책 및 향후조치계획을 200자 이내로 작성하세요.
기본규정을 우선적으로 참고하여, 이번 사례의 사고원인과 일치하는 사고원인이 있을 경우 해당 규정의 재발방지대책 및 향후조치계획을 그대로 작성하세요.  
기본규정에서 일치하는 사고원인이 없을 경우, 과거사례에서 유사한 사고 사례를 찾아 해당 사례의 재발방지대책 및 향후조치계획을 종합하여 작성하세요.  
제공된 정보 외에 새로운 정보를 추가하지 마세요.  
콤마로 구분된 일반 텍스트 형식으로 작성하며, Markdown 형식을 사용하지 마세요.

## 기본규정
사고원인: 기타 or 모름 or 알 수 없음 or 작업자의 과실 및 실수 or 작업자의 부주의 및 관리소홀
재방지대책 및 향후조치계획: 작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 조치 계획.

## 과거사례
사례1) 계절/기온: 여름/22°C
위치: 콘크리트펌프 바닥버림타설중
사고유형: 물체에 맞음
사고원인: 펌프카 붐대를 최대거리로 사용하면서 하중이 집중되어 아웃트리거 지반이 붕괴.
재발방지대책 및 향후조치계획: 관리적 대책으로 차량계 건설기계 작업계획서 작성 및 검토와 펌프카 안전점검 체크리스트에 의한 사전 점검, 기술적 대책으로 펌프카 아웃트리거 설치 시 지반상태 사전 점검 및 받침대 추가 설치 확인, 유도자 또는 신호수 배치, 작업구역 설정, 타 근로자 출입금지 조치, 교육적 대책으로 펌프카 작업 전 안전교육 실시 및 확인, 당 작업 전 TBM 시 작업내용 및 위험포인트 내용 공유, 향후 조치 계획으로 재발 방지 대책 수립 및 현장 관리 철저.
사례2) 계절/기온: 여름/24°C
위치: 콘크리트펌프 바닥
사고유형: 부딪힘
사고원인: 콘크리트 펌핑중 아웃리거가 흙바닥으로 밀리면서 흙바닥이 침하되어 펌프카가 기울어짐, 진동에 의한 아웃리거 변위
재발방지대책 및 향후조치계획: 펌프카 설치 위치 사전 검토와 아웃리거 변위 여부 점검을 통한 재발 방

In [39]:
print(predictions["generation"][0])

사고원인에 따라 재발방지대책 및 향후조치계획을 다음과 같이 마련합니다.

펌프카 아웃트리거 바닥 고임목을 3단으로 보강한 경우에도 지반 침하가 발생한 점을 고려하여, 사고원인과 유사한 사례 2번의 재발방지대책 및 향후조치계획을 참고하여 다음과 같이 마련합니다.

재발방지대책 및 향후조치계획: 펌프카 설치 위치 사전 검토와 아웃트리거 변위 여부 점검을 통한 재발 방지 대책 마련, 아웃트리거의 펼친 길이가 상이한 점을 확인하여 일관된 펼침 길이를 유지할 수 있도록 조치, 타설 위치가 건물 끝부분 모서리에 위치하는 것을 피하는 조치.


In [40]:
submission = pd.read_csv("dataset/rawdata/sample_submission.csv")
# submission

In [41]:
submission[CFG.target_var].iloc[:len(predictions["generation"])] = predictions["generation"]
embed = generation_evaluator.model.encode(predictions["generation"], batch_size=CFG.batch_size)
for i in range(embed.shape[1]):
    submission[f"vec_{i}"].iloc[:len(predictions["generation"])] = embed[:len(predictions["generation"]), i]
submission.to_csv(os.path.join(architecture_path, f"{CFG.model}_{CFG.model_version}_submission.csv"), index=False)
submission

/tmp/ipykernel_322199/233741819.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  submission[CFG.target_var].iloc[:len(predictions["generation"])] = predictions["generation"]
/tmp/ipykernel_322199/233741819.py:1: SettingWithCopyWarning: 
A

,ID,재발방지대책 및 향후조치계획,vec_0,vec_1,vec_2,vec_3,vec_4,vec_5,vec_6,vec_7,...,vec_758,vec_759,vec_760,vec_761,vec_762,vec_763,vec_764,vec_765,vec_766,vec_767
0,TEST_000,사고원인에 따라 재발방지대책 및 향후조치계획을 다음과 같이 마련합니다.\n\n펌프카...,-0.101560,-0.335588,0.204730,0.759485,-0.585241,-0.017222,0.910240,-0.133073,...,0.174485,0.543020,-0.220740,1.215412,0.596296,0.053567,-0.286832,-0.257518,0.058335,-0.023290
1,TEST_001,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.058341,-0.021910,-0.015262,0.147683,-0.186973,0.187705,0.811971,-0.510945,...,0.721078,1.620734,0.464886,1.790103,1.441870,-0.456865,0.148589,0.061933,1.051767,-0.065163
2,TEST_002,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.058341,-0.021910,-0.015262,0.147683,-0.186973,0.187705,0.811971,-0.510945,...,0.721078,1.620734,0.464886,1.790103,1.441870,-0.456865,0.148589,0.061933,1.051767,-0.065163
3,TEST_003,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.158147,-0.046215,0.133375,0.198483,-0.002719,0.509944,0.777665,-0.346766,...,0.129542,1.409083,0.341762,2.172928,1.008591,-0.733254,0.128938,0.013035,1.003701,-0.241921
4,TEST_004,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.456665,0.534332,0.026379,0.447033,-0.282548,0.423338,0.757060,-0.413785,...,0.543926,1.274405,-0.129869,1.076926,0.785648,-0.174213,-0.396689,0.121921,0.274014,-0.189515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959,TEST_959,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.205770,-0.067283,0.058336,0.157842,-0.635118,-0.014432,0.247918,-0.291136,...,0.973297,1.749823,0.300514,1.174022,1.469392,-0.544406,-0.028871,-0.224975,0.876777,0.060239
960,TEST_960,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.076335,-0.079766,0.091450,-0.009272,-0.496150,-0.118708,0.554688,-0.365446,...,0.903120,1.721105,0.227220,1.485778,1.311675,-0.572299,0.092219,-0.101023,1.033259,-0.115422
961,TEST_961,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.158147,-0.046215,0.133375,0.198483,-0.002719,0.509944,0.777665,-0.346766,...,0.129542,1.409083,0.341762,2.172928,1.008591,-0.733254,0.128938,0.013035,1.003701,-0.241921
962,TEST_962,작업 전 안전교육 실시와 안전관리자 안전점검 실시를 통한 재발 방지 대책 및 향후 ...,-0.158147,-0.046215,0.133375,0.198483,-0.002719,0.509944,0.777665,-0.346766,...,0.129542,1.409083,0.341762,2.172928,1.008591,-0.733254,0.128938,0.013035,1.003701,-0.241921
